# 시트명·하이퍼링크 테이블명 전환 + 최초일자→생성일 (Jupyter)

이미 `survey_v1` / `survey_v2` 로 만들어 둔 **완성 Excel 파일**을 입력받아 다음을 수행합니다.

1. 샘플 시트 이름을 `번호`(예: `1`,`2`) → **테이블명** 으로 변경
2. 하이퍼링크를 `번호` 셀에서 떼어내 **`테이블명` 셀**에 연결 (클릭 시 해당 시트로 이동)
3. `데이터보관최초일자` 컬럼을 **전부 그 테이블의 생성일**(`user_objects.created`)로 교체

> **원본은 절대 수정하지 않습니다.** `원본명_테이블명버전.xlsx` 새 파일을 만들어 거기에만 기록합니다.
> v1/v2 는 헤더의 `소재` 컬럼 유무로 **자동 판별**합니다 (셀 2에서 수동 지정도 가능).

| 셀 | 내용 | 비고 |
|---|---|---|
| 1 | 임포트 | |
| 2 | **설정 ★ 수정 필요** | 파일경로, DB 접속정보 |
| 3 | 유틸 함수 | 시트명 정제·셀 파싱 |
| 4 | 원본 Excel 읽기 + 버전 자동판별 | 번호·테이블명·소재 수집 |
| 5 | Oracle 접속 + 생성일 맵 조회 | ⏳ v1 은 PROD 만 필요 |
| 6 | 체크포인트 저장 | 생성일 맵 |
| 7 | 체크포인트 로드 (선택) | 재실행 시 셀 5 건너뛰기 |
| 8 | **새 파일 생성·변환·저장** | 시트명·링크·생성일 일괄 적용 |
| 9 | 검증 리포트 | 누락·중복 점검 |

In [ ]:
# 셀 1: 임포트
import os, re, shutil, pickle
from datetime import datetime

import oracledb
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import column_index_from_string

print('imports OK')

In [ ]:
# 셀 2: 설정 ★ 아래 값을 실제 환경에 맞게 수정하세요
ORACLE_CLIENT_LIB = r'C:\oracle\instantclient_21_9'  # ★

# survey 실행 때와 동일하게 입력 (v1 은 PROD 만 채우면 됨)
PROD_HOST     = 'PROD_HOST'      # ★
PROD_PORT     = 1521
PROD_SERVICE  = 'PROD_SERVICE'   # ★
PROD_USER     = 'PROD_USER'      # ★
PROD_PASSWORD = 'PROD_PASSWORD'  # ★

ARCH_HOST     = 'ARCHIVE_HOST'     # ★ v2 만
ARCH_PORT     = 1521
ARCH_SERVICE  = 'ARCHIVE_SERVICE'  # ★ v2 만
ARCH_USER     = 'ARCHIVE_USER'     # ★ v2 만
ARCH_PASSWORD = 'ARCHIVE_PASSWORD' # ★ v2 만

INFA_HOST     = 'INFA_HOST'      # ★ v2 만
INFA_PORT     = 1521
INFA_SERVICE  = 'INFA_SERVICE'   # ★ v2 만
INFA_USER     = 'INFA_USER'      # ★ v2 만
INFA_PASSWORD = 'INFA_PASSWORD'  # ★ v2 만

# survey 로 만든 완성 Excel 파일 (이 파일은 수정되지 않음)
SOURCE_EXCEL = r'C:\path\to\완성본.xlsx'  # ★

# survey 실행 때 쓴 번호 시작 셀과 동일하게
START_CELL = 'B4'

# 'auto' = 헤더의 '소재' 유무로 자동판별 / 강제하려면 'v1' 또는 'v2'
VERSION = 'auto'

CHECKPOINT_FILE = 'relink_created_checkpoint.pkl'

# 새 파일 경로 (원본명 + _테이블명버전)
OUTPUT_EXCEL = re.sub(r'\.xlsx$', '_테이블명버전.xlsx', SOURCE_EXCEL)
print(f'원본 : {SOURCE_EXCEL}')
print(f'출력 : {OUTPUT_EXCEL}')

In [ ]:
# 셀 3: 유틸 함수
# Excel 시트명 금지문자: \ / ? * [ ] :  / 최대 31자 / 작은따옴표는 링크와 충돌 → _ 치환
SHEET_BAD_RE = re.compile(r"[\\/?*\[\]:']")

def parse_cell(ref):
    m = re.match(r'^([A-Za-z]+)(\d+)$', ref.strip())
    if not m:
        raise ValueError(f'잘못된 셀 참조: {ref}')
    return column_index_from_string(m.group(1)), int(m.group(2))

def sanitize_sheet_name(name, used_lower):
    """Excel 규칙에 맞게 정제 + 중복 시 _2, _3 … 부여. used_lower 는 소문자 집합(in-place 갱신)."""
    s = SHEET_BAD_RE.sub('_', str(name)).strip()
    if not s:
        s = 'SHEET'
    s = s[:31]
    base = s
    i = 2
    while s.lower() in used_lower:
        suf = f'_{i}'
        s = base[:31 - len(suf)] + suf
        i += 1
    used_lower.add(s.lower())
    return s

def fmt_created(v):
    """user_objects.created → 'YYYY-MM-DD' (기존 데이터 포맷과 동일)"""
    if v is None:
        return None
    if isinstance(v, datetime):
        return v.strftime('%Y-%m-%d')
    return str(v)[:10]

print('utilities OK')

In [ ]:
# 셀 4: 원본 Excel 읽기 + 버전 자동판별 (읽기 전용 — 원본 변경 안 함)
if not os.path.isfile(SOURCE_EXCEL):
    raise FileNotFoundError(f'파일 없음: {SOURCE_EXCEL}')

start_col, start_row = parse_cell(START_CELL)

_wb_ro = load_workbook(SOURCE_EXCEL, read_only=True, data_only=True)
_ws_ro = _wb_ro.active
DATA_SHEET = _ws_ro.title

# ── 버전 판별: 헤더(데이터 시작 행 위쪽 1~5행)에서 '소재' 탐색 ──
def detect_version():
    if VERSION in ('v1', 'v2'):
        return VERSION
    for r in range(max(1, start_row - 5), start_row):
        for c in range(start_col, start_col + 10):
            val = _ws_ro.cell(row=r, column=c).value
            if val is not None and str(val).strip() == '소재':
                return 'v2'
    return 'v1'

ver = detect_version()

# 컬럼 오프셋 (start_col 기준)
if ver == 'v2':
    OFF_NO, OFF_LOC, OFF_TBL, OFF_DATE = 0, 2, 3, 6
else:
    OFF_NO, OFF_LOC, OFF_TBL, OFF_DATE = 0, None, 2, 5

# ── 레코드 수집: 번호 셀이 비면 종료 ──
records = []
r = start_row
while True:
    no_val = _ws_ro.cell(row=r, column=start_col + OFF_NO).value
    if no_val is None or str(no_val).strip() == '':
        break
    tbl = _ws_ro.cell(row=r, column=start_col + OFF_TBL).value
    loc = (_ws_ro.cell(row=r, column=start_col + OFF_LOC).value
           if OFF_LOC is not None else None)
    old_sheet = str(int(no_val)) if isinstance(no_val, float) and no_val.is_integer() \
                else str(no_val).strip()
    records.append({
        'excel_row':  r,
        'no':         no_val,
        'old_sheet':  old_sheet,
        'table_name': str(tbl).strip() if tbl is not None else '',
        'location':   str(loc).strip() if loc is not None else None,
    })
    r += 1

_wb_ro.close()
print(f'버전 판별: {ver}  (수동지정={VERSION})')
print(f'데이터 시트: {DATA_SHEET}')
print(f'레코드 {len(records)}개  (행 {start_row} ~ {start_row + len(records) - 1})')
print('예시:', records[:3])

In [ ]:
# 셀 5: Oracle 접속 + 테이블 생성일 맵 조회 (⏳ v1 은 PROD 만 필요)
oracledb.init_oracle_client(lib_dir=ORACLE_CLIENT_LIB)

CREATED_SQL = """
    SELECT object_name, created
    FROM   user_objects
    WHERE  object_type = 'TABLE'
"""

def load_created_map(user, password, dsn, label):
    try:
        conn = oracledb.connect(user=user, password=password, dsn=dsn)
    except Exception as e:
        print(f'{label} 접속 실패 (건너뜀): {e}')
        return {}
    try:
        cur = conn.cursor()
        cur.execute(CREATED_SQL)
        m = {row[0]: fmt_created(row[1]) for row in cur.fetchall()}
        cur.close()
        print(f'{label} 생성일 {len(m)}개')
        return m
    finally:
        conn.close()

prod_created = load_created_map(
    PROD_USER, PROD_PASSWORD, f'{PROD_HOST}:{PROD_PORT}/{PROD_SERVICE}', 'PROD')

arch_created, infa_created = {}, {}
if ver == 'v2':
    arch_created = load_created_map(
        ARCH_USER, ARCH_PASSWORD, f'{ARCH_HOST}:{ARCH_PORT}/{ARCH_SERVICE}', 'ARCHIVE')
    infa_created = load_created_map(
        INFA_USER, INFA_PASSWORD, f'{INFA_HOST}:{INFA_PORT}/{INFA_SERVICE}', 'INFA')

print(f'완료: PROD {len(prod_created)} / ARCH {len(arch_created)} / INFA {len(infa_created)}')

In [ ]:
# 셀 6: 체크포인트 저장 (DB 재조회 생략용)
with open(CHECKPOINT_FILE, 'wb') as f:
    pickle.dump({
        'prod_created': prod_created,
        'arch_created': arch_created,
        'infa_created': infa_created,
    }, f)
print(f'체크포인트 저장: {CHECKPOINT_FILE}')

In [ ]:
# 셀 7: 체크포인트 로드 (선택 — 재실행 시 셀 5 건너뛰고 이 셀만 실행)
with open(CHECKPOINT_FILE, 'rb') as f:
    _ck = pickle.load(f)
prod_created = _ck.get('prod_created', {})
arch_created = _ck.get('arch_created', {})
infa_created = _ck.get('infa_created', {})
print(f'로드 완료: PROD {len(prod_created)} / ARCH {len(arch_created)} / INFA {len(infa_created)}')

In [ ]:
# 셀 8: 새 파일 생성 → 시트명 변경 + 하이퍼링크 이동 + 생성일 교체 → 저장
# 원본을 새 경로로 복사한 뒤, 새 파일에만 기록 (원본 보존)
shutil.copy2(SOURCE_EXCEL, OUTPUT_EXCEL)
print(f'복사 완료: {OUTPUT_EXCEL}')

wb = load_workbook(OUTPUT_EXCEL)
ws = wb[DATA_SHEET]
LINK_FONT  = Font(color='0563C1', underline='single', bold=True)
PLAIN_FONT = Font()

def created_for(rec):
    """소재에 따라 알맞은 DB 의 생성일 선택 (없으면 다른 DB 폴백)"""
    loc = rec['location']
    if loc == 'INFA':
        order = (infa_created, prod_created, arch_created)
    elif loc == 'Archive Only':
        order = (arch_created, prod_created, infa_created)
    else:  # Combined / Prod Only / v1(소재없음)
        order = (prod_created, arch_created, infa_created)
    for m in order:
        if rec['table_name'] in m and m[rec['table_name']]:
            return m[rec['table_name']]
    return None

# 데이터 시트 등 비-샘플 시트 이름은 보존 대상 → 중복 방지 집합에 미리 등록
sample_old = {rec['old_sheet'] for rec in records}
used_lower = {t.lower() for t in wb.sheetnames if t not in sample_old}

renamed = missing_sheet = missing_created = 0
report  = []

for rec in records:
    old      = rec['old_sheet']
    tbl      = rec['table_name']
    xrow     = rec['excel_row']
    new_name = sanitize_sheet_name(tbl, used_lower)

    # 1) 시트명 변경 (번호 → 테이블명)
    if old in wb.sheetnames:
        wb[old].title = new_name
        renamed += 1
    else:
        missing_sheet += 1
        report.append(f'  [시트없음] 번호 {old} (테이블 {tbl}) — 링크는 그대로 연결')

    # 2) 하이퍼링크를 번호 셀 → 테이블명 셀로 이동
    no_cell  = ws.cell(row=xrow, column=start_col + OFF_NO)
    no_cell.hyperlink = None
    no_cell.font      = PLAIN_FONT

    tbl_cell = ws.cell(row=xrow, column=start_col + OFF_TBL)
    tbl_cell.hyperlink = f"#'{new_name}'!A1"
    tbl_cell.font      = LINK_FONT

    # 3) 데이터보관최초일자 → 테이블 생성일
    cdate = created_for(rec)
    if cdate:
        ws.cell(row=xrow, column=start_col + OFF_DATE).value = cdate
    else:
        missing_created += 1
        report.append(f'  [생성일없음] {tbl} ({rec["location"]}) — 기존 값 유지')

wb.save(OUTPUT_EXCEL)
print(f'저장 완료: {OUTPUT_EXCEL}')
print(f'시트명 변경 {renamed}개 / 시트누락 {missing_sheet}개 / 생성일누락 {missing_created}개')

In [ ]:
# 셀 9: 검증 리포트
print('=== 변환 요약 ===')
print(f'대상 레코드      : {len(records)}')
print(f'시트명 변경 성공 : {renamed}')
print(f'시트 누락        : {missing_sheet}')
print(f'생성일 누락      : {missing_created}')
print(f'원본(보존)       : {SOURCE_EXCEL}')
print(f'출력             : {OUTPUT_EXCEL}')
if report:
    print('\n--- 상세 (최대 50건) ---')
    for line in report[:50]:
        print(line)
    if len(report) > 50:
        print(f'  ... 외 {len(report) - 50}건')
else:
    print('\n경고 없음 — 전 건 정상 처리')